In [10]:
dataset = []

with open("data.txt", "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        line = line.strip()
        if not line:
            continue

        if "\t" not in line:
            print(f"Пропуск строки {i}: {line}")
            continue

        role, text = line.split("\t", 1)

        dataset.append({
            "id": len(dataset),
            "role": role.strip(),
            "text": text.strip()
        })

print(f"Всего строк: {len(dataset)}")
print(dataset[0])


Всего строк: 30
{'id': 0, 'role': 'HR', 'text': 'Чтобы оформить ежегодный оплачиваемый отпуск, сотрудник подает заявление не позднее чем за 14 календарных дней до начала отпуска. Руководитель подразделения рассматривает и согласовывает заявление, после чего отдел кадров издает приказ. Информация об отпуске вносится в кадровую систему и отражается в графике отпусков.'}


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

texts = [row["text"] for row in dataset]
embeddings = encoder.encode(texts, show_progress_bar=True)
embeddings = np.array(embeddings).astype("float32")

faiss.normalize_L2(embeddings)


/home/yugoff/Downloads/projects/my/laboration/project-lab-sp/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 1/1 [00:00<00:00,  2.40it/s]


In [ ]:
d = embeddings.shape[1]
index = faiss.IndexFlatIP(d)
index.add(embeddings)

print(f"В индексе {index.ntotal} векторов")


В индексе 30 векторов


In [ ]:
import json


faiss.write_index(index, "roles_index.faiss")

with open("dataset.json", "w", encoding="utf-8") as f:
    json.dump(dataset, f, ensure_ascii=False, indent=2)


In [14]:
def search(query, role, k=5):
    q_emb = encoder.encode([query])
    q_emb = np.array(q_emb).astype("float32")
    faiss.normalize_L2(q_emb)

    scores, ids = index.search(q_emb, 20)

    results = []
    for idx, score in zip(ids[0], scores[0]):
        doc = dataset[idx]
        if doc["role"] == role:
            results.append((doc["text"], score))
        if len(results) == k:
            break

    return results


In [15]:
for text, score in search("Как оформить отпуск?", role="HR"):
    print(score, text)


0.6607936 Отпуск по уходу за ребенком оформляется на основании заявления сотрудника и подтверждающих документов. HR фиксирует период отпуска в системе и контролирует соблюдение сроков выхода на работу.
0.5594393 Оформление командировки включает подачу заявки, согласование с руководителем и утверждение бюджета. HR издает приказ о командировке, а после возвращения сотрудник предоставляет отчет и подтверждающие документы.
0.55298764 Онбординг нового сотрудника включает оформление доступа к системам, ознакомление с внутренними политиками и назначение наставника. В течение испытательного срока HR отслеживает адаптацию сотрудника и собирает обратную связь от руководителя.
0.48950645 Аттестация сотрудников проводится в соответствии с утвержденным графиком и включает оценку компетенций и результатов работы. По итогам аттестации принимаются решения о развитии, обучении или изменении условий труда.
0.48349428 Обработка персональных данных сотрудников осуществляется в соответствии с требованиями 